In [1]:
import pandas as pd
import numpy as np




## Load Data

In [3]:
import os
# Data Paths
DATA_SUBFOLDER = ""
TRAIN_FILE_PATH = os.path.join(DATA_SUBFOLDER, "train.csv")
TEST_FILE_PATH = os.path.join(DATA_SUBFOLDER, "test.csv")

try:
    df_test_received  = pd.read_csv(TEST_FILE_PATH)
    df_train_received = pd.read_csv(TRAIN_FILE_PATH)
except FileNotFoundError:
    print("Error: train.csv or test.csv not found. Please ensure they are in the correct path.")
    raise # Stop execution if files are not found




## Data Split to make them < 100MB

###  Geting File Sizes

In [4]:
# Parametry
MAX_CHUNK_MB = 95
CHUNK_FOLDER = "DATA_chunks_by_size"
os.makedirs(CHUNK_FOLDER, exist_ok=True)

def split_csv_by_size(df, original_path, prefix, max_mb=MAX_CHUNK_MB):
    # Oblicz rozmiar pełnego pliku
    full_size_bytes = os.path.getsize(original_path)
    full_size_mb = full_size_bytes / (1024 * 1024)
    total_rows = len(df)

    # Oblicz ile wierszy zmieści się w jednym pliku
    rows_per_chunk = int((max_mb / full_size_mb) * total_rows)
    print(f"{prefix}: pełny rozmiar {full_size_mb:.2f} MB, wierszy: {total_rows}, wierszy na chunk: {rows_per_chunk}")

    # Podział i zapis
    num_chunks = (total_rows + rows_per_chunk - 1) // rows_per_chunk
    for i in range(num_chunks):
        start = i * rows_per_chunk
        end = min(start + rows_per_chunk, total_rows)
        chunk = df.iloc[start:end]
        filename = os.path.join(CHUNK_FOLDER, f"{prefix}_{i+1}.csv")
        chunk.to_csv(filename, index=False)
        chunk_size_mb = os.path.getsize(filename) / (1024 * 1024)
        print(f"Zapisano {filename} ({len(chunk)} wierszy, ~{chunk_size_mb:.2f} MB)")


# Podziel i zapisz
split_csv_by_size(df_train_received, "train.csv", "train")
split_csv_by_size(df_test_received, "test.csv", "test")
print("koniec")

train: pełny rozmiar 632.08 MB, wierszy: 11504798, wierszy na chunk: 1729154
Zapisano DATA_chunks_by_size\train_1.csv (1729154 wierszy, ~95.53 MB)
Zapisano DATA_chunks_by_size\train_2.csv (1729154 wierszy, ~96.59 MB)
Zapisano DATA_chunks_by_size\train_3.csv (1729154 wierszy, ~96.59 MB)
Zapisano DATA_chunks_by_size\train_4.csv (1729154 wierszy, ~96.59 MB)
Zapisano DATA_chunks_by_size\train_5.csv (1729154 wierszy, ~96.59 MB)
Zapisano DATA_chunks_by_size\train_6.csv (1729154 wierszy, ~96.95 MB)
Zapisano DATA_chunks_by_size\train_7.csv (1129874 wierszy, ~64.19 MB)
test: pełny rozmiar 413.82 MB, wierszy: 7669866, wierszy na chunk: 1760773
Zapisano DATA_chunks_by_size\test_1.csv (1760773 wierszy, ~96.68 MB)
Zapisano DATA_chunks_by_size\test_2.csv (1760773 wierszy, ~96.68 MB)
Zapisano DATA_chunks_by_size\test_3.csv (1760773 wierszy, ~96.68 MB)
Zapisano DATA_chunks_by_size\test_4.csv (1760773 wierszy, ~96.68 MB)
Zapisano DATA_chunks_by_size\test_5.csv (626774 wierszy, ~34.41 MB)


In [ ]:
MAX_TOTAL_MB = 95

# Getting Files Sizes
train_size_bytes = os.path.getsize(TRAIN_FILE_PATH)
test_size_bytes = os.path.getsize(TEST_FILE_PATH)

# Getting Row count
rows_in_train = len(df_train_received)
rows_in_test = len(df_test_received)

# MB Conversion
train_size_mb = train_size_bytes / (1024 * 1024)
test_size_mb = test_size_bytes / (1024 * 1024)

split_proporcion = train_size_mb/(test_size_mb + train_size_mb)

new_size_target_train  = split_proporcion * colective_size
new_size_target_test = colective_size - new_size_target_train

new_rows_ammount_train = int((new_size_target_train * rows_in_train)/train_size_mb)
new_rows_ammount_test = int((new_size_target_test * rows_in_test)/test_size_mb)

print(f"Rozmiar train.csv: {train_size_mb:.2f} MB rows {rows_in_train} new amount of lines { new_rows_ammount_train}")
print(f"Rozmiar test.csv: {test_size_mb:.2f} MB rows {rows_in_test} new amount of lines { new_rows_ammount_test}")

Wyliczenie proporcji wierszy by uzyskać 99 MB rozmiar pliku

In [ ]:
# Train target column
TARGET_COLUMN = 'Response'

if TARGET_COLUMN in df_train_received.columns:
    print('TARGET_COLUMN found')
else:
    print('ERROR !!! TARGET_COLUMN not found !!!\n', df_train_received.columns)


df_train_reduced, _ = train_test_split(
    df_train_received,
    train_size=50000,
    stratify=df_train_received["target"],
    random_state=42
)
df_train_received: pełny DataFrame z wszystkimi kolumnami
train_size=50000: wybierasz 50 tys. wierszy
stratify=...: zachowujesz proporcje klas w kolumnie "target"
df_train_reduced: zawiera wszystkie kolumny, tylko mniej wierszy
_: pozostałe wiersze, których nie używasz

In [ ]:
NEW_DATA_SUBFOLDER = 'DATA_reduced_to_size_under_100_MB'
os.makedirs(NEW_DATA_SUBFOLDER, exist_ok=True)
NEW_TRAIN_FILE_PATH = os.path.join(NEW_DATA_SUBFOLDER, "train.csv")
NEW_TEST_FILE_PATH = os.path.join (NEW_DATA_SUBFOLDER, "test.csv")

from sklearn.model_selection import train_test_split

# Zmniejszamy zbiór do 50 tys. wierszy, zachowując proporcje targetu
df_train_reduced, _ = train_test_split(
    df_train_received,
    train_size= new_rows_ammount_train,
    stratify=df_train_received[TARGET_COLUMN],
    random_state=42
)

# Zapisz do nowego pliku
df_train_reduced.to_csv(NEW_TRAIN_FILE_PATH, index=False)

# Ustal liczbę wierszy , np. 30 tys. new_rows_ammount_test


# Losowe pobranie wierszy z testu
df_test_reduced = df_test_received.sample(n=new_rows_ammount_test, random_state=42)

# Zapisz do nowego pliku
df_test_reduced.to_csv(NEW_TEST_FILE_PATH, index=False)


# Getting Files Sizes
train_size_bytes = os.path.getsize(NEW_TRAIN_FILE_PATH)
test_size_bytes = os.path.getsize(NEW_TEST_FILE_PATH)

# Getting Row count
rows_in_train = len(df_train_received)
rows_in_test = len(df_test_received)

# MB Conversion
train_size_mb = train_size_bytes / (1024 * 1024)
test_size_mb = test_size_bytes / (1024 * 1024)

colective_size =train_size_mb + test_size_mb
print(f"Rozmiar train_reduced.csv: {train_size_mb:.2f} MB rows {rows_in_train}")
print(f"Rozmiar test_reduced.csv: {test_size_mb:.2f} MB rows {rows_in_test}")
print(f"Rozmiar SUM: {colective_size:.2f}")